# LIGHT version: the action model only (~5 minutes)

Use this when the live run reads the football detector's CACHED detections (computed on the Mac before the run):
it skips the detector (its own environment + TensorRT engine build, ~15 extra minutes).
For the Colab detector (the "Later tests" in docs/MVP3_LIVE_TEST_PLAN.md) use `specialist_server.ipynb`.
No Roboflow key needed here. Steps: Runtime -> T4 GPU -> Run all -> paste the URL into Studio -> Live -> Test.

# Story Reader — specialist server (Colab GPU)

This notebook runs TWO models on a free Colab GPU behind ONE public URL:
- the **football detector** (Roboflow `football-players-detection-3zvbc/11`: players, goalkeepers, referees, ball)
  for MVP 3's real live test: 3 cameras x 5 frames per second while the match plays (the Mac is too slow for that);
- the **ball action spotting** model
[`OpenSportsLab/OSL-loc-snbas-2025-e2e`](https://huggingface.co/OpenSportsLab/OSL-loc-snbas-2025-e2e)
(12 classes: PASS, DRIVE, HEADER, HIGH PASS, OUT, CROSS, THROW IN, SHOT, BALL PLAYER BLOCK,
PLAYER SUCCESSFUL TACKLE, FREE KICK, GOAL) on a free Colab GPU, and gives you a public URL that the
dashboard on your Mac connects to.

## How to run (5 steps)
1. **Turn on the GPU:** menu *Runtime → Change runtime type → T4 GPU → Save*.
2. **Roboflow key (for the detector):** click the 🔑 *Secrets* icon (left bar) → *Add new secret* → name
   `ROBOFLOW_API_KEY` (exactly), value = your Roboflow API key, switch *Notebook access* on. Without it the
   action model still runs, only the detector is off.
3. *(Optional, only for the action model alone; the detector needs far more than ngrok's free 1 GB/month)* **ngrok token:** make a free account at https://dashboard.ngrok.com, copy your
   authtoken, then in Colab click the 🔑 *Secrets* icon (left bar) → *Add new secret* → name
   `NGROK_AUTHTOKEN`, value = your token, and switch *Notebook access* on.
   By default the notebook uses a free Cloudflare quick tunnel (no account, no data cap); ngrok only with TUNNEL = 'ngrok' in step 5.
4. **Run everything:** menu *Runtime → Run all*. It takes ~20–25 minutes (installing, downloading the models, and
   building the detector's TensorRT engine, which makes it 8x faster). Leave it alone until the URL box appears.
5. **Copy the URL** printed in big letters at the end (it looks like `https://xxxx.ngrok-free.app`
   or `https://xxxx.trycloudflare.com`).
6. **Paste it** into the dashboard: MVP 3 http://localhost:8000/studio → ● Live → *Colab URL* → *Test*
   (both the action model and the detector should show), or MVP 2 http://localhost:8000/story → *Connect*.

Keep this tab open while you use the dashboard. Colab stops the server after ~90 minutes of no activity
(free tier); then just *Run all* again and paste the new URL.

**License note:** the model is AGPL-3.0 (research / non-commercial use).

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: Runtime -> Change runtime type -> T4 GPU'
!pip install -q opensportslib==0.3.1 fastapi uvicorn pyngrok websockets huggingface_hub pyyaml
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. The server code (same as `colab/specialist_server.py` in the project)

## 3. Load the model, warm it up, and self-test
Downloads `config.yaml` + `model.pt` (150 MB) from Hugging Face. The self-test runs the model on fake frames and checks the speed: the **real-time factor** (inference time ÷ 2 s stride) must be < 1.

In [ ]:
import time, numpy as np, cv2
from specialist_server import Spotter, Session, make_app
spotter = Spotter()
print(spotter.health())
print('warm-up (first run is slow):', round(spotter.warm_up(), 2), 's')
times = [spotter.warm_up() for _ in range(3)]
print(f'inference on a {spotter.clip_len}-frame window: {np.mean(times)*1000:.0f} ms '
      f'-> real-time factor {np.mean(times)/2.0:.3f} (must be < 1)')
# Self-test of the live session logic with synthetic JPEG frames
session = Session(spotter, {'min_confidence': 0.3, 'stride_s': 2.0})
ok, jpg = cv2.imencode('.jpg', np.random.randint(0, 255, (224, 398, 3), np.uint8))
for i in range(30):
    due = session.add(i / spotter.fps, jpg.tobytes())
result = session.infer()
print('self-test OK:', {k: result[k] for k in ('frames', 'infer_ms', 'rtf', 'window')})

## 4. Start the server (in the background)

In [ ]:
import threading, uvicorn, requests
PORT = 8765
server = uvicorn.Server(uvicorn.Config(make_app(spotter), host='0.0.0.0', port=PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print('local /health:', requests.get(f'http://127.0.0.1:{PORT}/health').json()['model'])

## 5. Open a public tunnel and print the URL
Default: a free Cloudflare quick tunnel (no account, no data cap). The detector sends ~19 Mbit/s (~7 GB per half), far over ngrok's free 1 GB/month: set `TUNNEL = 'ngrok'` only for the action model alone (it uses the `NGROK_AUTHTOKEN` secret).

In [ ]:
import re, subprocess
from IPython.display import HTML, display
TUNNEL = 'cloudflare'   # or 'ngrok'
public_url = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN') if TUNNEL == 'ngrok' else None
except Exception:
    token = None
if token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = token
    public_url = ngrok.connect(PORT, 'http').public_url.replace('http://', 'https://')
else:
    print('Cloudflare quick tunnel (no account needed).')
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
    # start_new_session: stopping another cell (■) must not kill the tunnel
    tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    for line in tunnel.stdout:
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if found:
            public_url = found.group(0)
            break
    threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep reading its log
    time.sleep(5)
for attempt in range(10):   # check /health THROUGH the tunnel
    try:
        r = requests.get(public_url + '/health', headers={'ngrok-skip-browser-warning': '1'}, timeout=10)
        print('/health through the tunnel:', r.status_code, r.json()['gpu']); break
    except Exception as error:
        print('waiting for the tunnel...', type(error).__name__); time.sleep(3)
display(HTML(f'<div style="font-size:28px;font-weight:bold;padding:16px;border:3px solid #22c55e;'
             f'border-radius:8px">Paste this into the dashboard:<br><code>{public_url}</code></div>'))
print(public_url)

## 6. Keep this running
This cell prints a heartbeat every minute so you can see the server is alive. Stop it with the ■ button when you are done.

In [ ]:
while True:
    print(time.strftime('%H:%M:%S'), 'server alive at', public_url)
    time.sleep(60)